# Module 33 — Exercise 1: Cache-Aside Pattern with TTL and Invalidation

In this exercise, you will implement the Cache-Aside (Lazy-Loading) pattern with time-to-live expiration, key eviction, and cache invalidation on write mutations.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 33 README |



# Your turn


### Task 1: Implement `CacheAsideStore`

Implement a class `CacheAsideStore` wrapping a mock database and an in-memory cache:
- `get(key)`: check cache first; if cache miss, read from database, populate cache with TTL, and return value.
- `set(key, value)`: update database, and invalidate (evict) key from cache.


In [ ]:
# ANSWER 1
import time

class CacheAsideStore:
    def __init__(self, db: dict, ttl: float = 60.0):
        self.db = dict(db)
        self.cache: dict[str, tuple[object, float]] = {}  # key -> (val, expire_at)
        self.ttl = ttl
        self.db_reads = 0

    def get(self, key: str):
        now = time.time()
        if key in self.cache:
            val, expire_at = self.cache[key]
            if now < expire_at:
                return val
            del self.cache[key]  # expired
        
        if key in self.db:
            self.db_reads += 1
            val = self.db[key]
            self.cache[key] = (val, now + self.ttl)
            return val
        return None

    def set(self, key: str, value):
        self.db[key] = value
        if key in self.cache:
            del self.cache[key]  # Invalidate on write



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

store = CacheAsideStore({"user:1": "Alice", "user:2": "Bob"}, ttl=10.0)

# First read: Cache Miss -> DB Read
val1 = store.get("user:1")
# Second read: Cache Hit -> No DB Read
val2 = store.get("user:1")
db_reads_after_two = store.db_reads

# Mutation: Invalidate cache
store.set("user:1", "Alice Updated")
val3 = store.get("user:1")  # Re-reads from DB

results = [
    check(val1 == "Alice" and val2 == "Alice", "Task 1: Correct value returned"),
    check(db_reads_after_two == 1, "Task 1: Second read hit cache without DB access"),
    check(val3 == "Alice Updated" and store.db_reads == 2, "Task 1: Mutation invalidated cache and triggered reload"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

